# 2 Byte-Pair Encoding (BPE) Tokenizer

## 2.1 Unicode Standard

In [8]:
# a), b), c) chr(0) unicode is Null

print(chr(0))

try:
    assert chr(0) == Null
except:
    print('chr(0) representation is diff from:', """ ' "" ' """)
    
print("This is a test" + chr(0) + "string")

 
chr(0) representation is diff from:  ' "" ' 
This is a test string


## 2.2 Unicode Byte Encoding

In [9]:
# Unicode Vocab Size is roughly 150k and very sparse -> Better use Byte (0-255 integers) representation of text
# Every possible text sequence has a unique byte encoding decomposition - list of integers from 0 to 255 - and is perfectly decoded into the original text
# UTF-8 encoding is denser than UTF-16 / UTF-32 
# -> Input Text has a fixed amount of information: higher entropy for UTF-8 representation (less sparse, less 0s)

text = "Hello! こんにちは!"
print("Unicode Representation of the input text:", list(map(ord, text)), '\nlen:', len(list(map(ord, text))))
print("Byte Representation:", list(text.encode('utf-8')), "\nlen:", len(list(text.encode('utf-8'))))
print("Byte Representation:", list(text.encode('utf-16')), "\nlen:", len(list(text.encode('utf-16'))))
print("Byte Representation:", list(text.encode('utf-32')), "\nlen:", len(list(text.encode('utf-32'))))


Unicode Representation of the input text: [72, 101, 108, 108, 111, 33, 32, 12371, 12435, 12395, 12385, 12399, 33] 
len: 13
Byte Representation: [72, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33] 
len: 23
Byte Representation: [255, 254, 72, 0, 101, 0, 108, 0, 108, 0, 111, 0, 33, 0, 32, 0, 83, 48, 147, 48, 107, 48, 97, 48, 111, 48, 33, 0] 
len: 28
Byte Representation: [255, 254, 0, 0, 72, 0, 0, 0, 101, 0, 0, 0, 108, 0, 0, 0, 108, 0, 0, 0, 111, 0, 0, 0, 33, 0, 0, 0, 32, 0, 0, 0, 83, 48, 0, 0, 147, 48, 0, 0, 107, 48, 0, 0, 97, 48, 0, 0, 111, 48, 0, 0, 33, 0, 0, 0] 
len: 56


In [10]:
# This function decodes the input byte sring byte by byte. It will be incorrect for unicode characters that encode to several bytes

def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
 return "".join([bytes([b]).decode("utf-8") for b in bytestring])

decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

try:
    decode_utf8_bytes_to_str_wrong("こんにちは".encode("utf-8"))
except: print("Wrong decoding")

# Not every byte-sequence can be decoded into a text string / Unicode Character

Wrong decoding


## 2.4 BPE Tokenizer Training

In [11]:
# First step is pre-tokenization of the training corpus: avoids semantic split (e.g. dog. / dog! / dog, have very similar semantic meaning)
# -> Use regex library to split individual words inside the training corpus. BPE is done with words barriers
# BPE: Merge highest frequency byte-pairs into a newly created vocab token (idx)
# Each merge adds one token to the vocab idx -> Vocab Size grows from 256 to 256 + num_merges
# Special Tokens are being added to the vocab during a separated step - outside of the tokenizer training
# -> these strings should never be split into chuncks (e.g. '<|endoftext|>' is one token)

In [12]:
from pretokenization_example import find_chunk_boundaries

with open('../data/TinyStoriesV2-GPT4-valid.txt', 'rb') as f:
    num_processes = 4
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")
    print(boundaries)
    # The following is a serial implementation, but you can parallelize this
    # by sending each start/end pair to a set of processes.
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        f.seek(start)
    chunk = f.read(end - start).decode("utf-8", errors="ignore")
    print(len(chunk))
    # Run pre-tokenization on your chunk and store the counts for each pre-token
    

[0, 5625758, 11252559, 16877372, 22502601]
5622918


In [89]:
import regex as re
from collections import Counter

# Hyperparameters
text = '''
<|startoftext|> u don't have to be scared of the loud dog, I'll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.
<|endoftext|>
Once upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad.
Tom asked his friend, Sam, to help him search for the ball. They looked high and low, but they could not find the ball. Tom said, "I think my ball fell into the pit."
Sam and Tom went close to the pit. They were scared, but they wanted to find the red ball. They looked into the pit, but it was too dark to see. Tom said, "We must go in and search for my ball."
They went into the pit to search. It was dark and scary. They could not find the ball. They tried to get out, but the pit was too deep. Tom and Sam were stuck in the pit. They called for help, but no one could hear them. They were sad and scared, and they never got out of the pit.
<|endoftext|>'''

## Pre-tokenizer

def pre_tokenizer(text: str, special_tokens: list) -> dict[tuple[int, ...], int]:
    dict_special_tokens = set(special_tokens)

    # First step is isolating special tokens -> re.escape (for handling <| inside special tokens) + re.split
    # Seperate the special tokens from the rest of the chunks: [Doc1] + <|endoftext|> + [Doc2]
    doc_list = re.split('(' + '|'.join(list(map(re.escape, list(special_tokens)))) + ')', text)

    # Second step: separating words inside each none special token chunk + encoding those words into bytes int representation
    # Use re.finditer(pattern, string) / print(m.group(), m.start(), m.end())
    pre_tokenized_text = {}
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""" # From OpenAI tokenizer repository
    for doc in doc_list:
        if doc not in dict_special_tokens:
            for m in re.finditer(PAT, doc):
                encoded_word = tuple(m.group().encode('utf-8'))
                if encoded_word not in pre_tokenized_text:
                    pre_tokenized_text[encoded_word] = 1
                else:
                    pre_tokenized_text[encoded_word] += 1
                

    return pre_tokenized_text

pre_tokenized_text = pre_tokenizer(text, special_tokens = ['<|endoftext|>', '<|startoftext|>'])
print(pre_tokenized_text)

## BPE

def get_pairs_from_word(word: list):
    pairs = {}
    for i in range(len(word) - 1):
        pair = (word[i], word[i+1])
        if pair not in pairs:
            pairs[pair] = 1
        else: pairs[pair] += 1
    return pairs

def get_all_pairs(pre_tokenized_text: dict[tuple[int, ...], int]) -> dict[tuple[int, ...], int]:
    total_pairs = Counter({})
    for word, multiply in pre_tokenized_text.items():
        pairs = Counter(get_pairs_from_word(list(word)))
        for key in pairs:
            pairs[key] *= multiply

        total_pairs += pairs

    return total_pairs

# To get the dict of {pairs tuple: frequencies}
full_pairs = get_all_pairs(pre_tokenized_text=pre_tokenized_text)
print(full_pairs)

# Let's create the vocab
vocab = {}
for i in range(256):
    vocab[i] = bytes([i])
# vocab is 0-255 original bytes + 256 - 256 + len(special_tokens) of special tokens bytes encoding stream
special_tokens = ['<|endoftext|>', '<|startoftext|>']
for i, token_str in enumerate(special_tokens):
    vocab[256 + i] = token_str.encode('utf-8')

# Let's do one merge
merge_pair = max(full_pairs, key=full_pairs.get)
vocab[len(vocab)] = bytes(merge_pair)
assert tuple(vocab[len(vocab)-1]) == merge_pair


{(10,): 7, (32, 117): 1, (32, 100, 111, 110): 1, (39, 116): 1, (32, 104, 97, 118, 101): 1, (32, 116, 111): 9, (32, 98, 101): 1, (32, 115, 99, 97, 114, 101, 100): 3, (32, 111, 102): 2, (32, 116, 104, 101): 15, (32, 108, 111, 117, 100): 1, (32, 100, 111, 103): 1, (44,): 14, (32, 73): 1, (39, 108, 108): 1, (32, 112, 114, 111, 116, 101, 99, 116): 1, (32, 121, 111, 117): 1, (34, 46): 1, (32, 84, 104, 101): 2, (32, 109, 111, 108, 101): 3, (32, 102, 101, 108, 116): 1, (32, 115, 111): 1, (32, 115, 97, 102, 101): 2, (32, 119, 105, 116, 104): 1, (32, 108, 105, 116, 116, 108, 101): 2, (32, 103, 105, 114, 108): 1, (46,): 20, (32, 83, 104, 101): 1, (32, 119, 97, 115): 6, (32, 118, 101, 114, 121): 2, (32, 107, 105, 110, 100): 1, (32, 97, 110, 100): 10, (32, 115, 111, 111, 110): 1, (32, 99, 97, 109, 101): 1, (32, 116, 114, 117, 115, 116): 1, (32, 104, 101, 114): 2, (32, 72, 101): 2, (32, 108, 101, 97, 110, 101, 100): 1, (32, 97, 103, 97, 105, 110, 115, 116): 1, (32, 115, 104, 101): 1, (32, 107, 101, 

In [5]:
# Serializing and saving

import json

vocab = {'a': 1, 'b': 2, 'c': 3}

with open('../data/vocab.json', 'w') as f:
    f.write(json.dumps(vocab))

with open('../data/vocab.json', 'r') as f:
    vocab = json.loads(f.read())
    print(type(vocab))


<class 'dict'>


## 2.5 train_BPE TBD
## 2.6. Encoding and Decoding TBD

In [6]:
# Import HugginFace tokenizer, encode the training dataset, serialize and save the sequence of tokens into numpy array of datatype uint16.



In [14]:
import numpy as np

array = np.array([1,2,3,5,6,4], dtype=np.uint16)

np.save('../data/array.npy', array)

In [22]:
array = np.load('../data/array.npy')

print(type(array[0]))

<class 'numpy.uint16'>
